# Bibliotecas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from src import ppd, utils, imputation as fimp
from src.MSCVAE import MSCVAE

DIR_DATA = "data/"

# Parâmetros gerais e dados do processo

In [ ]:
timestamp = "Timestamp"
process_name = "processo_A"
base_name = process_name + ".parquet"
subsistema = DIR_DATA + process_name + "_subsistema.csv"

# Carregando dados

In [ ]:
drop_outliers = False

list_colomuns_drop = ["TIMESTAMP", 'TAG_028', 'TAG_027', 'TAG_036']
df_dataset = ppd.load_dataset_principal(DIR_DATA+base_name, list_colomuns_drop, timestamp,dropna=False, use_chunks=False, chunksize=10000)

df_dataset["TAG_037"] = df_dataset["TAG_037"].ffill().bfill()
df_dataset["TAG_041"] = df_dataset["TAG_041"].ffill().bfill()
df_dataset["TAG_042"] = df_dataset["TAG_042"].ffill().bfill()
df_dataset["TAG_043"] = df_dataset["TAG_043"].ffill().bfill()
df_dataset["TAG_038"] = df_dataset["TAG_038"].ffill().bfill()
df_dataset["TAG_039"] = df_dataset["TAG_039"].ffill().bfill()
df_dataset["TAG_040"] = df_dataset["TAG_040"].ffill().bfill()

df_dataset.dropna(inplace=True)
df_dataset

# Parâmetros de ON/OFF

In [ ]:
pre_process = []
pp_var_ref_desligado = "TAG_031"
pp_valor_ref_desligado = 0.5
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 90
pp_pos_corte_transitorio = 270
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# Importação das tags e descrições

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,subsistema)

In [ ]:
df_sistema

In [ ]:
df_sistema_drop

# Seleção do período de treinamento e criação do modelo

In [ ]:
df_train_list = []

## Período 1

In [ ]:
#Data train 

# train 1
start_date_train = pd.to_datetime('2026-08-30 00:00:00')
end_date_train = pd.to_datetime('2026-08-31 10:20:00')

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)

list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train.reset_index(inplace=True, drop=True)

print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)

df_train_list.append(df_train)

## Período 2

In [ ]:
#Data train 
start_date_train2 = pd.to_datetime('2026-09-02 03:00:00')
end_date_train2 = pd.to_datetime('2026-09-02 13:20:00')


mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)

list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train2.reset_index(inplace=True, drop=True)

print(df_train2.shape)


eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

df_train_list.append(df_train2)

## Período 3

In [ ]:
#Data train 
start_date_train3 = pd.to_datetime('2026-09-09 17:00:00') 
end_date_train3 = pd.to_datetime('2026-09-11 13:20:00')

mask = (df_dataset[timestamp] > start_date_train3) & (df_dataset[timestamp] <= end_date_train3)
df_train3 = df_dataset.loc[mask]

# pre - processamento
print(df_train3.shape)

list_periods_train = []
for pro in pre_process:
    df_train3,_,list_aux = ppd.drop_transitorio_desligado(df_train3,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train3.reset_index(inplace=True, drop=True)

print(df_train3.shape)


eixoX_train3 = df_train3[timestamp]
df_train3 = df_train3.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train3], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train3], ignore_index=True)

df_train_list.append(df_train3)

## Período 4

In [ ]:
#Data train 
start_date_train4 = pd.to_datetime('2026-09-21 12:00:00') 
end_date_train4 = pd.to_datetime('2026-09-22 09:20:00')

mask = (df_dataset[timestamp] > start_date_train4) & (df_dataset[timestamp] <= end_date_train4)
df_train4 = df_dataset.loc[mask]

# pre - processamento
print(df_train4.shape)

list_periods_train = []
for pro in pre_process:
    df_train4,_,list_aux = ppd.drop_transitorio_desligado(df_train4,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train4.reset_index(inplace=True, drop=True)

print(df_train4.shape)


eixoX_train4 = df_train4[timestamp]
df_train4 = df_train4.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train4], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train4], ignore_index=True)

df_train_list.append(df_train4)

## Período 5

In [ ]:
#Data train 
start_date_train5 = pd.to_datetime('2026-09-22 16:00:00') 
end_date_train5 = pd.to_datetime('2026-09-23 04:10:00')

mask = (df_dataset[timestamp] > start_date_train5) & (df_dataset[timestamp] <= end_date_train5)
df_train5 = df_dataset.loc[mask]

# pre - processamento
print(df_train5.shape)

list_periods_train = []
for pro in pre_process:
    df_train5,_,list_aux = ppd.drop_transitorio_desligado(df_train5,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train5.reset_index(inplace=True, drop=True)

print(df_train5.shape)


eixoX_train5 = df_train5[timestamp]
df_train5 = df_train5.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train5], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train5], ignore_index=True)

for _ in range(15):
    df_train_list.append(df_train5)

## Período 6

In [ ]:
#Data train 
start_date_train6 = pd.to_datetime('2026-09-21 00:00:00')
end_date_train6 = pd.to_datetime('2026-09-28 00:00:00')

mask = (df_dataset[timestamp] > start_date_train6) & (df_dataset[timestamp] <= end_date_train6)
df_train6 = df_dataset.loc[mask]

# pre - processamento
print(df_train6.shape)

list_periods_train = []
for pro in pre_process:
    df_train6,_,list_aux = ppd.drop_transitorio_desligado(df_train6,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train6.reset_index(inplace=True, drop=True)

print(df_train6.shape)


eixoX_train6 = df_train6[timestamp]
df_train6 = df_train6.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train6], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train6], ignore_index=True)

for _ in range(10):
    df_train_list.append(df_train6)

## Período N

In [ ]:
# #Data train 
# start_date_trainN = pd.to_datetime('YYYY-mm-dd HH:MM:SS')
# end_date_trainN = pd.to_datetime('YYYY-mm-dd HH:MM:SS')

# mask = (df_dataset[timestamp] > start_date_trainN) & (df_dataset[timestamp] <= end_date_trainN)
# df_trainN = df_dataset.loc[mask]

# # pre - processamento
# print(df_trainN.shape)

# list_periods_train = []
# for pro in pre_process:
#     df_trainN,_,list_aux = ppd.drop_transitorio_desligado(df_trainN,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
#     list_periods_train = [*list_periods_train,*list_aux]

# df_trainN.reset_index(inplace=True, drop=True)

# print(df_trainN.shape)


# eixoX_trainN = df_trainN[timestamp]
# df_trainN = df_trainN.drop(timestamp,axis=1)

# df_train = pd.concat([df_train, df_trainN], ignore_index=True)
# eixoX_train = pd.concat([eixoX_train, eixoX_trainN], ignore_index=True)

# df_train_list.append(df_trainN)

## Drop colunas treinamento
Executar caso uma variável foi utilizada como referência mas não faz parte do modelo e foi utilizada apenas para remover os desligados

In [ ]:
list_columns_drop_train = ["TAG_031"]
df_train_list = [df.drop(columns=list_columns_drop_train, errors="ignore") for df in df_train_list]
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_train_list[0], subsistema)

## Criação do Modelo

In [ ]:
mscvae = MSCVAE()

gain = 0.55
epochs = 20
mscvae.fit(df_train_list, gain=gain, epochs=epochs, verbose=True)

# Seleção do período para testar o modelo

In [ ]:
start_date = pd.to_datetime("2026-07-01 00:00:00")
end_date = pd.to_datetime("2026-12-30 00:00:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

list_periods_nan = ppd.select_periods_nan(df_dataset,timestamp)

# pre - processamento
print(df_test.shape)

list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
    
print(df_test.shape)

df_test_aux = df_test.copy()
eixoX_test = df_test[timestamp]
df_test = df_test.drop(timestamp,axis=1)

df_test.reset_index(inplace=True, drop=True)

## Drop colunas teste

In [ ]:
## Executar caso uma variável foi utilizada como referência mas não faz parte do modelo
list_columns_drop_teste = list_columns_drop_train
df_test = df_test.drop(columns=list_columns_drop_teste, errors="ignore")
df_test_aux = df_test_aux.drop(columns=list_columns_drop_teste, errors="ignore")

# Imputação de valores faltantes
Testa métodos de imputação treinados com **os mesmos períodos de treino do MSCVAE** e aplicados no período de teste antes da predição. Dois eventos são cobertos:
- **Outlier extremo (falha de sensor)**: eventos reais do período de teste. Só entram o valor absurdo para o sensor, alcançado num salto que ele nunca deu no treino, e o sensor que vai abruptamente para o fundo de escala (`LIMITES_ESCALA`). Desvio de processo, mesmo grande, vai cru para o MSCVAE, para não mascarar o início de uma falha.
- **TAG com NaN**: lacunas **injetadas sinteticamente** em uma cópia do teste. O valor real é conhecido, então dá para medir o erro. Os NaN do carregamento continuam sendo removidos como antes.

A aplicação segue a regra do preprocessing de produção:
1. outlier vira NaN;
2. lacuna menor que `START_MINUTES` com valor anterior válido é preenchida com ffill;
3. o restante vai para o `IterativeImputer`;
4. linha com ≥ `MAX_BAD_RATIO` das variáveis inválidas não é reconstruída.

## Parâmetros

In [ ]:
# Estimadores do IterativeImputer (edite o dict para incluir/remover)
estimadores = fimp.estimadores_padrao(random_state=0)

# Limpeza do treino do imputer: linha com |x - mediana| > FATOR * escala robusta sai do treino
FATOR_FALHA_GROSSEIRA = 50

# Outlier extremo (falha de sensor) no teste: só dispara quando as duas condições valem
FATOR_EXTREMO = 3.0          # |x - mediana| > FATOR * max(faixa p1-p99, |mediana|) do treino
SALTO_EXTREMO = 20.0         # e o valor chegou lá num salto > SALTO * maior salto normal (p99.9) do treino
K_RETORNO = 1.0              # após disparar, marca até o valor voltar para [p1 - K*faixa, p99 + K*faixa]

# Fundo de escala por TAG: (limite inferior, limite superior) do instrumento; None = sem limite daquele lado.
# Valores em que o sensor travou nos dados deste processo. Confirmar com o range do instrumento.
LIMITES_ESCALA = {
    "TAG_001": (-81.1, None),    # vai a -81.1 e volta (6 eventos no histórico)
    "TAG_014": (-58.0, None),    # vai a -58 e o histórico grava uma rampa interpolada até voltar
    "TAG_006": (None, 33.3),     # picos de 1 amostra em 33.3, valor máximo das 3 TAGs
    "TAG_007": (None, 33.3),
    "TAG_008": (None, 33.3),
    "TAG_010": (None, 1036.25),  # patamar no máximo do histórico (fora do período de teste)
}
TOL_ESCALA = 0.05            # zona do limite: a menos de 5% da distância entre a faixa normal e o limite
SALTO_ESCALA = 3.0           # e chegou lá num salto > SALTO * maior salto normal (p99.9), em até 2 amostras

# Regra de aplicação (igual à produção)
START_MINUTES = 10           # lacuna menor que isso, com âncora -> ffill; senão -> imputer
MAX_BAD_RATIO = 0.7          # linha com >= 70% das variáveis inválidas não é reconstruída

# Lacunas NaN sintéticas
LACUNAS_POR_TAG = 3
DUR_MIN, DUR_MAX = 10, 240   # minutos
FRAC_SIMULTANEAS = 0.3       # fração das lacunas que derruba 2-3 TAGs juntas
SEED = 42

PASSO_AMOSTRAGEM = 1         # > 1 subamostra o treino dos imputers (mais rápido)
N_EVENTOS_PLOT = 6           # gráficos antes x depois (metade NaN, metade outlier)
N_DESVIOS_PLOT = 4           # gráficos de conferência dos desvios mantidos

## Treino dos imputers
Usa os períodos do `df_train`, sem as repetições de peso do `df_train_list` e sem os timestamps duplicados de períodos sobrepostos. As linhas com falha grosseira de sensor são removidas, porque senão o imputer aprende a reproduzir a falha.

Cada método passa pelo `train_imputer` de produção: holdout temporal com lacunas de 10–240 min, uma TAG por vez. O `.pkl` fica em `./models/`, e o `blocos_imputer[metodo]` segue o formato do bloco `imputer` usado no deploy.

In [ ]:
cols_modelo = list(mscvae.generator.mean.index)

df_train_imputer = pd.concat([eixoX_train.rename(timestamp), df_train[cols_modelo]], axis=1)
df_train_imputer = df_train_imputer.drop_duplicates(subset=timestamp).sort_values(timestamp).reset_index(drop=True)
df_train_imputer, falhas_treino = fimp.limpar_falhas_treino(df_train_imputer, cols_modelo, fator=FATOR_FALHA_GROSSEIRA)
print(df_train_imputer.shape)
falhas_treino

In [ ]:
imputers, blocos_imputer, tempos_imputer = fimp.treinar_imputers(df_train_imputer, timestamp, process_name, estimadores, passo_amostragem=PASSO_AMOSTRAGEM)

## Comparação no holdout do treino
MAE normalizado pelo desvio-padrão da TAG no treino, o que permite comparar TAGs de escalas diferentes. O `.pkl (MB)` importa no deploy, porque o runtime baixa o imputer a cada ciclo.

In [ ]:
stats_treino = fimp.estatisticas_outlier(df_train_imputer[cols_modelo], ts=df_train_imputer[timestamp])
tabela_holdout, r2_holdout, mae_norm_holdout = fimp.resumo_holdout(blocos_imputer, stats_treino["std"], tempos_imputer)
tabela_holdout

In [ ]:
fimp.plot_metricas(tabela_holdout, ["MAE norm. (mediana)", "R² (mediana)", "tempo treino (s)", ".pkl (MB)"], "Holdout do treino").show()
fimp.plot_heatmap_tags(r2_holdout, "R² por TAG no holdout do treino", zmin=0, zmax=1).show()

## Detecção de outliers extremos (falha de sensor)
Só é imputado o que é falha de sensor. Desvio de processo, mesmo grande, vai cru para o MSCVAE. São duas regras.

**Valor absurdo**: o valor cumpre **as duas** condições:
1. **Absurdo para o nível da TAG**: |x − mediana| > `FATOR_EXTREMO` × max(faixa p1–p99, |mediana|) do treino. Como o nível da TAG entra na escala, uma queda a zero (parada, trip) não passa de ~1×.
2. **Alcançado num salto**: |x[t] − x[t−1]| > `SALTO_EXTREMO` × o maior salto normal (p99,9) da TAG no treino. Falha de processo evolui aos poucos e não cumpre essa condição, por maior que fique.

**Fundo de escala**: só para as TAGs de `LIMITES_ESCALA`. Cobre a falha que fica perto da faixa operacional, onde a regra anterior não chega (um sensor a ~1000 que vai para −58 tem fator ~1). O valor cumpre **as duas** condições:
1. **Na zona do limite**: a menos de `TOL_ESCALA` da distância entre a faixa normal (p1/p99) e o limite.
2. **Chegou lá de forma abrupta**: em até 2 amostras, um salto > `SALTO_ESCALA` × o maior salto normal. O processo que sobe aos poucos até saturar o instrumento é desvio real e fica cru.

Depois do disparo, a marcação segue só para frente:
- no valor absurdo, até o valor voltar para a faixa normal (± `K_RETORNO` faixas);
- no fundo de escala, enquanto o valor estiver mais perto do limite do que da faixa normal. Isso pega a rampa interpolada que o histórico grava depois da falha, mas não a parada de equipamento logo depois de um pico.

O que vem antes do disparo nunca é marcado, como no runtime, que só vê o passado. Por isso a primeira amostra da transição (a média do minuto em que o sensor travou) fica crua.

A tabela compara com o envelope (k=2) e com a regra MAD da produção (`robust_z`), que marcam também desvio de processo.

### Candidatos a fundo de escala
Valores em que o sensor parou depois de sair abruptamente da faixa normal, repetidos em 2 ou mais eventos do histórico, para ajudar a montar `LIMITES_ESCALA`. É só sugestão: TAG quantizada (valor inteiro) e parada de equipamento também aparecem. Confirme com o range do instrumento antes de incluir.

In [ ]:
fimp.candidatos_fundo_escala(df_dataset[cols_modelo], stats_treino)

In [ ]:
df_test_modelo = df_test[cols_modelo].reset_index(drop=True)
ts_test = eixoX_test.reset_index(drop=True)
descricao_tag = df_sistema.set_index("VARIAVEL")["DESC"].to_dict()

fimp.comparar_regras_outlier(df_test_modelo, stats_treino, fator=FATOR_EXTREMO, salto=SALTO_EXTREMO, k_retorno=K_RETORNO,
                            limites_escala=LIMITES_ESCALA, tol_escala=TOL_ESCALA, salto_escala=SALTO_ESCALA)

In [ ]:
mask_outlier = fimp.detectar_outliers(df_test_modelo, stats_treino, fator=FATOR_EXTREMO, salto=SALTO_EXTREMO, k_retorno=K_RETORNO,
                                     limites_escala=LIMITES_ESCALA, tol_escala=TOL_ESCALA, salto_escala=SALTO_ESCALA)
print(f"Pontos marcados como outlier: {int(mask_outlier.values.sum())} em {int(mask_outlier.any().sum())} TAGs")

### Conferência: desvios fora da faixa normal
Cada trecho fora de [p1 − K·faixa, p99 + K·faixa], por TAG. `imputado` = contém outlier extremo; `mantido` = vai cru para o MSCVAE.

Os gráficos mostram os desvios mantidos mais perto do limite e os mais longos (faixa verde = faixa normal do treino; faixa laranja = trecho do desvio), para conferir que nenhum desvio de processo foi apagado.

In [ ]:
desvios = fimp.listar_desvios(df_test_modelo, ts_test, stats_treino, mask_outlier, k_retorno=K_RETORNO)
display(desvios.groupby("tratamento").agg(trechos=("tag", "size"), amostras=("amostras", "sum"), TAGs=("tag", "nunique")))
desvios

In [ ]:
for _, dv in fimp.selecionar_desvios_mantidos(desvios, N_DESVIOS_PLOT).iterrows():
    fimp.plot_desvio_mantido(dv["tag"], ts_test, df_test_modelo, stats_treino, dv["inicio"], dv["fim"], fator=FATOR_EXTREMO,
                             salto=SALTO_EXTREMO, descricao=descricao_tag.get(dv["tag"]),
                             limites=LIMITES_ESCALA.get(dv["tag"])).show()

## Injeção sintética de lacunas NaN
As lacunas são colocadas longe dos outliers reais, para que a predição de referência fique limpa nesses trechos.

In [ ]:
df_test_falhas, mask_sintetico = fimp.injetar_lacunas(df_test_modelo, ts_test, lacunas_por_tag=LACUNAS_POR_TAG, dur_min=DUR_MIN, dur_max=DUR_MAX,
                                                     frac_simultaneas=FRAC_SIMULTANEAS, evitar=mask_outlier, seed=SEED)
print(f"Células NaN injetadas: {int(mask_sintetico.values.sum())} em {int(mask_sintetico.any().sum())} TAGs")

## Aplicação da imputação
`ffill (baseline)` preenche tudo com ffill/bfill. Os demais métodos seguem a regra de produção.

In [ ]:
df_imputados, infos_imputacao = {}, {}
for nome, imputer in {"ffill (baseline)": None, **imputers}.items():
    df_imputados[nome], infos_imputacao[nome] = fimp.aplicar_imputacao(df_test_falhas, ts_test, imputer, mask_outlier,
                                                                        start_minutes=START_MINUTES, max_bad_ratio=MAX_BAD_RATIO)
fimp.resumo_imputacao(infos_imputacao)

## Erro nas lacunas sintéticas (valor real conhecido)

In [ ]:
tabela_lacunas, metricas_lacunas_tag = fimp.metricas_lacunas(df_test_modelo, df_imputados, mask_sintetico, stats_treino["std"])
tabela_lacunas

In [ ]:
fimp.plot_metricas(tabela_lacunas, ["MAE norm. (mediana)", "RMSE norm. (mediana)", "R² (mediana)"], "Lacunas sintéticas no teste").show()
fimp.plot_heatmap_tags(metricas_lacunas_tag.pivot(index="tag", columns="método", values="MAE norm."),
                       "MAE normalizado por TAG nas lacunas sintéticas (menor é melhor)", zmin=0, zmax=1, escala="RdYlGn_r").show()

## Visualização: TAGs antes × depois da imputação
- **Linha de cima:** a TAG como chegou, com a lacuna NaN (faixa cinza) ou o outlier (faixa e marcadores vermelhos).
- **Linha de baixo:** a TAG depois da imputação, uma cor por método, com o valor real tracejado nas lacunas sintéticas.

Clique na legenda para ligar ou desligar métodos.

In [ ]:
eventos_imputacao = fimp.listar_eventos(mask_sintetico, mask_outlier, ts_test, df_test_modelo)
eventos_imputacao

In [ ]:
for _, ev in fimp.selecionar_eventos(eventos_imputacao, N_EVENTOS_PLOT).iterrows():
    fimp.plot_imputacao_tag(ev["tag"], ts_test, df_test_falhas, df_imputados, mask_sintetico, mask_outlier, df_verdade=df_test_modelo,
                            inicio=ev["inicio"], fim=ev["fim"], descricao=descricao_tag.get(ev["tag"])).show()

In [ ]:
# Para ver um evento específico, use uma linha de `eventos_imputacao`:
# ev = eventos_imputacao.iloc[0]
# fimp.plot_imputacao_tag(ev["tag"], ts_test, df_test_falhas, df_imputados, mask_sintetico, mask_outlier, df_verdade=df_test_modelo,
#                         inicio=ev["inicio"], fim=ev["fim"], margem_min=120).show()

# Predição

In [ ]:
predictions = mscvae.predict(df_test, timestamps=eixoX_test)

## Gráfico de Detecção

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict((predictions["phi"]/mscvae.threshold), predictions["timestamp"], (mscvae.threshold/mscvae.threshold))
fig_all_period.show()

## Predição com imputação
- **Referência**: a predição acima, com os dados carregados, os outliers reais crus e sem lacunas sintéticas.
- **Sem imputação (dropna)**: o comportamento atual diante das lacunas, que descarta as linhas com NaN e deixa os outliers crus.
- **ffill (baseline)** e **um cenário por método**: as linhas que continuaram NaN (bloqueadas) são descartadas antes do predict.

In [ ]:
predicoes = {"Referência": predictions}
for nome, df_cenario in {"Sem imputação (dropna)": df_test_falhas, **df_imputados}.items():
    df_pred, ts_pred = fimp.preparar_predicao(df_cenario, ts_test)
    print(f"{nome}: {len(df_pred)} linhas")
    predicoes[nome] = mscvae.predict(df_pred, timestamps=ts_pred)

## Efeito da imputação no índice de anomalia
- **% alarmes**: fração dos instantes com phi/limiar > 1.
- **cobertura**: fração dos instantes afetados pelo evento que ainda têm predição. O dropna perde monitoramento nas lacunas.
- **|Δ phi/limiar|**: desvio em relação à referência dentro das lacunas sintéticas.

Nos outliers reais, o esperado é que a imputação derrube os alarmes causados pela falha do sensor.

In [ ]:
efeito_predicao = fimp.resumo_efeito_predicao(predicoes, mscvae.threshold, ts_test, mask_sintetico, mask_outlier, janela=mscvae.generator.w)
efeito_predicao

In [ ]:
fimp.plot_phi_comparacao(predicoes, mscvae.threshold, ts_test, mask_sintetico, mask_outlier, freq="5min").show()